In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load the dataset
df = pd.read_csv("equipment_data_1000.csv", parse_dates=["Timestamp"])
print("Original Data:")
df.head()

Original Data:


,Timestamp,Equipment_ID,Sensor1,Sensor2,Sensor3,Failure
0,2025-01-01 00:00:00,EQ_004,75.546974,0.529875,112.658262,0
1,2025-01-01 00:05:00,EQ_005,78.628833,0.424821,120.819956,1
2,2025-01-01 00:10:00,EQ_003,77.405046,0.457364,124.572804,0
3,2025-01-01 00:15:00,EQ_005,76.119420,0.614845,134.558223,0
4,2025-01-01 00:20:00,EQ_005,71.047628,0.511327,127.045065,0


In [3]:
# Sort by Equipment_ID and Timestamp
df.sort_values(by=["Equipment_ID", "Timestamp"], inplace=True)

In [4]:
# Set Timestamp as index for time-based features
df.set_index("Timestamp", inplace=True)

In [5]:
# Step 1: Time-based features
df['Hour'] = df.index.hour
df['DayOfWeek'] = df.index.dayofweek
df['IsWeekend'] = df['DayOfWeek'].isin([5, 6]).astype(int)

In [6]:
# Step 2: Rolling statistics (window=3 and 5) per equipment
rolling_features = ['Sensor1', 'Sensor2', 'Sensor3']
for sensor in rolling_features:
    df[f'{sensor}_roll_mean_3'] = df.groupby('Equipment_ID')[sensor].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
    df[f'{sensor}_roll_std_3']  = df.groupby('Equipment_ID')[sensor].transform(lambda x: x.rolling(window=3, min_periods=1).std())
    df[f'{sensor}_roll_mean_5'] = df.groupby('Equipment_ID')[sensor].transform(lambda x: x.rolling(window=5, min_periods=1).mean())
    df[f'{sensor}_roll_std_5']  = df.groupby('Equipment_ID')[sensor].transform(lambda x: x.rolling(window=5, min_periods=1).std())

In [7]:
# Step 3: Lag features
for sensor in rolling_features:
    df[f'{sensor}_lag_1'] = df.groupby('Equipment_ID')[sensor].shift(1)
    df[f'{sensor}_lag_2'] = df.groupby('Equipment_ID')[sensor].shift(2)

In [8]:
# Step 4: Change (delta) features
for sensor in rolling_features:
    df[f'{sensor}_diff_1'] = df[sensor] - df[f'{sensor}_lag_1']

In [9]:
# Drop rows with NaN from rolling or lagging (optional: depends on your model)
df.dropna(inplace=True)

In [10]:
# Reset index
df.reset_index(inplace=True)

In [16]:
# Final preview
print("\nEngineered Feature Sample:")
df.head()


Engineered Feature Sample:


,Timestamp,Equipment_ID,Sensor1,Sensor2,Sensor3,Failure,Hour,DayOfWeek,IsWeekend,Sensor1_roll_mean_3,...,Sensor3_roll_std_5,Sensor1_lag_1,Sensor1_lag_2,Sensor2_lag_1,Sensor2_lag_2,Sensor3_lag_1,Sensor3_lag_2,Sensor1_diff_1,Sensor2_diff_1,Sensor3_diff_1
0,2025-01-01 02:00:00,EQ_001,73.321077,0.502422,138.093063,1,2,2,0,70.595054,...,7.851997,68.524606,69.939478,0.393247,0.466522,123.979270,136.999574,4.796470,0.109175,14.113793
1,2025-01-01 02:45:00,EQ_001,78.481032,0.758956,129.022772,0,2,2,0,73.442238,...,6.716023,73.321077,68.524606,0.502422,0.393247,138.093063,123.979270,5.159955,0.256534,-9.070292
2,2025-01-01 03:10:00,EQ_001,87.866799,0.533848,113.421654,0,3,2,0,79.889636,...,10.150651,78.481032,73.321077,0.758956,0.502422,129.022772,138.093063,9.385767,-0.225108,-15.601117
3,2025-01-01 03:25:00,EQ_001,74.879375,0.631760,128.902475,0,3,2,0,80.409068,...,8.996716,87.866799,78.481032,0.533848,0.758956,113.421654,129.022772,-12.987424,0.097911,15.480821
4,2025-01-01 03:45:00,EQ_001,72.265705,0.488193,111.049302,0,3,2,0,78.337293,...,11.483151,74.879375,87.866799,0.631760,0.533848,128.902475,113.421654,-2.613669,-0.143567,-17.853173


In [17]:
# Save engineered dataset (optional)
df.to_csv("engineered_equipment_data.csv", index=False)